In [ ]:
import numpy as np
from lib.vqe import *
from lib.dvr2d import *
from lib.pot_gen import *
from lib.utils import *
from lib.local_utils import *
import matplotlib.pyplot as plt

%matplotlib inline

plt.rcParams["figure.figsize"] = (8,6)
plt.rcParams["font.size"] = 14
plt.rcParams["figure.dpi"] = 300

%load_ext autoreload

In [ ]:
mol_params = arhcl_params
pot_file = base_dir + 'arhcl.txt'

dvr_options = {
    'type': 'jacobi',
    'N_R': 35, 
    'N_theta': 4,
    'l': 1,
    'K_max': 0,
    'r_min': 3.4,
    'r_max': 5, 
    'trunc': 0,
    'J': 0
}

vqe_options = {
    # 'optimizers': ['SLSQP.1000'],
    'optimizers': ['COBYLA.500'],
    'repeats': 1, 
    'seed': 42
}

ansatz_options = {
    'type': 'greedyent',
    'layers': 5,
    'reps': 2,
    'num_qubits': 5
}

pot = get_pot(pot_file, take_cos=True)
dvr_vqe = DVR_VQE(mol_params, pot, log_dir=scratch_dir + 'test/')

In [ ]:
def build_circuit(ansatz_options, removed_gates):
    from qiskit import QuantumCircuit
    from qiskit.circuit import Parameter

    num_qubits = ansatz_options['num_qubits']
    reps = ansatz_options['reps']
    ansatz = QuantumCircuit(num_qubits)

    tri_inds = np.triu_indices(5, k=1)
    num_gates = len(tri_inds[0])
    
    p = 0
    for i in range(reps):
        for q in range(num_qubits):
            theta = Parameter(f'$x_{p}$')
            p += 1
            ansatz.ry(theta, q)
        ansatz.barrier(range(5))
        for j, (q1, q2) in enumerate(zip(*tri_inds)):
            if j + i * num_gates not in removed_gates:
                ansatz.cx(q1, q2)
        ansatz.barrier(range(5))
    for q in range(num_qubits):
        theta = Parameter(f'$x_{{{p}}}$')
        p += 1
        ansatz.ry(theta, q)
    return ansatz

ansatz = build_circuit(ansatz_options, [])
ansatz.draw(output='mpl', plot_barriers=False)

In [ ]:
%autoreload
dvr_vqe.get_dvr_vqe(dvr_options, ansatz_options, vqe_options)

In [ ]:
def vqe_optimize(h_dvr_pauli, ansatz, optimizers, repeat=3, return_graphs=True, verbose=False):
    # from qiskit.algorithms.optimizers import NELDER_MEAD
    # optimizers = [SLSQP(maxiter=600)]
    converge_cnts = np.empty([len(optimizers)], dtype=object)
    converge_vals = np.empty([len(optimizers)], dtype=object)

    params = None
    # params = np.array([0.0 for i in range(ansatz1.num_parameters)])
    for i, optimizer in enumerate(optimizers):
        if verbose:
            print('Optimizer: {}        '.format(type(optimizer).__name__))
        algorithm_globals.random_seed = 42

        def store_intermediate_result(eval_count, parameters, mean, std):
            counts.append(eval_count)
            values.append(mean)
            if verbose:
                print(f'\r{eval_count}, {mean}', end='')
        
        best_res = None
        best_count = None
        for j in range(repeat):
            counts = []
            values = []
            vqe = VQE(ansatz, optimizer, callback=store_intermediate_result, initial_point=params, 
                    quantum_instance=QuantumInstance(backend=Aer.get_backend('statevector_simulator')))
            result = vqe.compute_minimum_eigenvalue(operator=h_dvr_pauli)
            if verbose:
                print()

            if (best_res is None) or (values[-1] < best_res):
                best_res = values[-1]
                best_count = counts[-1]
                converge_cnts[i] = np.asarray(counts)
                converge_vals[i] = np.asarray(values)
    if verbose:
        print('\nOptimization complete      ')
    if return_graphs:
        return converge_cnts, converge_vals
    else:
        return best_count, best_res

In [ ]:
max_layers = 8
num_qubits = 5
samples = 50

optimizers = [SLSQP(maxiter=1000)]
best_vals = []
best_counts = []
best_circs = []
for l in range(max_layers):
    print(f'Layer: {l + 1}')
    layer_list = [np.random.randint(0, 4, (num_qubits, 1), ) for i in range(samples)]
    if l == 0:
        circ_list = layer_list
    else:
        circ_list = [np.concatenate([best_circs[-1], l], axis=-1) for l in layer_list]
    
    best_val_layer = None
    for c in circ_list:
        ansatz = build_circuit(c, opt_level=2, add_h=True, add_rs=False)
        if ansatz.num_parameters > 0:
            count, val = vqe_optimize(h_dvr_pauli, ansatz, optimizers, return_graphs=False)
            if (best_val_layer is None) or (val < best_val_layer):
                print(val)
                best_val_layer = val
                best_count_layer = count
                best_circ_layer = c
    print('--------------')
    print(best_val_layer)
    print(best_count_layer)
    print(build_circuit(best_circ_layer, opt_level=2, add_h=True, add_rs=False))
    print('--------------')

    best_vals.append(best_val_layer)
    best_counts.append(best_count_layer)
    best_circs.append(best_circ_layer)
    


In [ ]:
print(best_circs[-1])
print(best_vals)
# plt.plot(best_vals)
best_depths = list(map(lambda c: build_circuit(c, add_rs=False, add_h=True).depth(), best_circs))
print(best_depths)
best_num_params = list(map(lambda c: build_circuit(c, add_rs=False, add_h=True).num_parameters, best_circs))
print(best_num_params)
# print(build_circuit(best_circs[-1]))
qc = build_circuit(best_circs[-1], add_rs=False, add_h=True)
print(qc.depth())
qc.draw(output='mpl')

In [ ]:
ref = -115.26788457220933
plt.plot(best_depths, abs(ref - np.array(best_vals)))
plt.scatter(15, 0.22017660276553386)
plt.scatter(9, 1.2068774694549376)
plt.axhline(abs(ref - eigvals[0]), ls='--', c='blue', label='DVR ground state')
plt.xlabel('Circuit depth')
plt.ylabel('Energy difference from converged ground state (cm$^{-1}$)')
# plt.title('Energy convergence for various optimizers')
plt.yscale('log')

In [ ]:
ref = -115.26788457220933
plt.plot(best_num_params, abs(ref - np.array(best_vals)))
plt.scatter(15, 0.22017660276553386)
plt.scatter(10, 1.2068774694549376)
plt.axhline(abs(ref - eigvals[0]), ls='--', c='blue', label='DVR ground state')
plt.xlabel('Circuit depth')
plt.ylabel('Energy difference from converged ground state (cm$^{-1}$)')
# plt.title('Energy convergence for various optimizers')
plt.yscale('log')

In [ ]:
import pylab

ref = -115.26788457220933
# print(converge_cnts1, converge_vals1)
pylab.rcParams['figure.figsize'] = (12, 8)
for i, optimizer in enumerate(optimizers):
    print(type(optimizer).__name__, converge_vals1[i][-1], abs(ref - converge_vals1[i][-1]))
    pylab.plot(converge_cnts1[i], converge_vals1[i], label=type(optimizer).__name__)
# pylab.axhline(eigvals[0])

print(abs(ref - eigvals[0]))
pylab.axhline(eigvals[0], ls='--', c='blue', label='Numerical g.s.')
pylab.axhline(ref, ls='--', c='black', label='Converged g.s.')
pylab.xlabel('Eval count')
pylab.ylabel('Energy (cm$^{-1}$)')
# pylab.title('Energy convergence for various optimizers')
# pylab.xlim(800, 1000)
pylab.ylim(-116, -80)
pylab.legend(loc='upper right');

In [ ]:
# pylab.rcParams['figure.figsize'] = (12, 8)
for i, optimizer in enumerate(optimizers):
    # pylab.plot(converge_cnts1[i][:], abs(ref - converge_vals1[i][:]), label=type(optimizer).__name__)
    pylab.plot(converge_cnts1[i][:], abs(ref - converge_vals1[i][:]), label=type(optimizer).__name__)
pylab.axhline(abs(ref - eigvals[0]), ls='--', c='blue', label='DVR ground state')
pylab.xlabel('Eval count')
pylab.ylabel('Energy difference from converged ground state (cm$^{-1}$)')
# pylab.title('Energy convergence for various optimizers')
pylab.yscale('log')
# plt.ylim(0,1)
pylab.legend(loc='upper right');
# print(abs(ref - converge_vals1[:, -1]))

In [ ]:
ansatz2 = TwoLocal(5, rotation_blocks=['ry', 'rz', 'ry'], entanglement_blocks='cx', entanglement='full', reps=3).decompose()
ansatz2.draw(output='mpl', fold=-1)
# print(ansatz)

In [ ]:
h_dvr_pauli = pauli_decompose(h_dvr)

np.allclose(h_dvr, h_dvr_pauli.to_matrix())

print(h_dvr.shape)
print(h_dvr_pauli.coeffs.shape)
# print(h_dvr_pauli)

In [ ]:
optimizers = [COBYLA(maxiter=8000), L_BFGS_B(maxfun=8000), SLSQP(maxiter=1000), NELDER_MEAD(maxfev=8000)]
# from qiskit.algorithms.optimizers import NELDER_MEAD
# optimizers = [SLSQP(maxiter=600)]
converge_cnts2 = np.empty([len(optimizers)], dtype=object)
converge_vals2 = np.empty([len(optimizers)], dtype=object)

repeat = 3
params = None
# params = np.array([0.0 for i in range(ansatz1.num_parameters)])
for i, optimizer in enumerate(optimizers):
    print('Optimizer: {}        '.format(type(optimizer).__name__))
    algorithm_globals.random_seed = 42

    def store_intermediate_result(eval_count, parameters, mean, std):
        counts.append(eval_count)
        values.append(mean)
        print(f'\r{eval_count}, {mean}', end='')
    
    best_res2 = None
    for j in range(repeat):
        counts = []
        values = []
        vqe = VQE(ansatz2, optimizer, callback=store_intermediate_result, initial_point=params, 
                quantum_instance=QuantumInstance(backend=Aer.get_backend('statevector_simulator')))
        result = vqe.compute_minimum_eigenvalue(operator=h_dvr_pauli)
        print()

        if (best_res2 is None) or (values[-1] < best_res2):
            best_res2 = values[-1]
            converge_cnts2[i] = np.asarray(counts)
            converge_vals2[i] = np.asarray(values)
print('\nOptimization complete      ')

In [ ]:
import pylab

ref = -115.26788457220933
# print(converge_cnts2, converge_vals2)
pylab.rcParams['figure.figsize'] = (12, 8)
for i, optimizer in enumerate(optimizers):
    print(type(optimizer).__name__, converge_vals2[i][-1], abs(ref - converge_vals2[i][-1]))
    pylab.plot(converge_cnts2[i], converge_vals2[i], label=type(optimizer).__name__)
# pylab.axhline(eigvals[0])

print(abs(ref - eigvals[0]))
pylab.axhline(eigvals[0], ls='--', c='blue', label='Numerical g.s.')
pylab.axhline(ref, ls='--', c='black', label='Converged g.s.')
pylab.xlabel('Eval count')
pylab.ylabel('Energy (cm$^{-1}$)')
# pylab.title('Energy convergence for various optimizers')
# pylab.xlim(800, 1000)
pylab.ylim(-116, -80)
pylab.legend(loc='upper right');

In [ ]:
# pylab.rcParams['figure.figsize'] = (12, 8)
for i, optimizer in enumerate(optimizers):
    # pylab.plot(converge_cnts1[i][:], abs(ref - converge_vals1[i][:]), label=type(optimizer).__name__)
    pylab.plot(converge_cnts2[i][:], abs(ref - converge_vals2[i][:]), label=type(optimizer).__name__)
pylab.axhline(abs(ref - eigvals[0]), ls='--', c='blue', label='DVR ground state')
pylab.xlabel('Eval count')
pylab.ylabel('Energy difference from converged ground state (cm$^{-1}$)')
# pylab.title('Energy convergence for various optimizers')
pylab.yscale('log')
# plt.ylim(0,1)
pylab.legend(loc='upper right');
# print(abs(ref - converge_vals1[:, -1]))